# Construct BOW, DTM, TFIDF, TFIDF_L2

BOW → DTM → TFIDF → TFIDF_L2, plus the DFIDF computation back-joined to VOCAB.

This is also you declare your bag level and your significant-word selection principle for TFIDF_L2

Note: I have 22 book_ids but 12 true novels. The stories from the collection "poirot investigates" are split into individual book_ids.

I am choosing Chapter as my bag because it is a good narrative WORD that isn't too big...

I am following along with my M07 HW as I complete this section.

## Setup

### Import Libraries

In [1]:
# import libraries
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, TfidfTransformer

### Configuration

In [2]:
# specify OHCO and bags
OHCO = ['book_id', 'chap_num', 'para_num', 'sent_num', 'token_num']

bags = dict(
    SENTS = OHCO[:4],
    PARAS = OHCO[:3],
    CHAPS = OHCO[:2],
    BOOKS = OHCO[:1]
)

In [ ]:
# set chapter as bag
bag = "CHAPS"

### Load Data

In [6]:
# load in tables
CORPUS = pd.read_csv('data/CORPUS.csv', sep='\t').set_index(OHCO)
VOCAB = pd.read_csv('data/VOCAB.csv', sep='\t').set_index('term_str')

In [17]:
# testing
CORPUS.groupby(bags[bag]).size()

book_id                       chap_num
giants-bread                  0           1879
                              1           2297
                              2           1298
                              3           3255
                              4           3953
                                          ... 
the-seven-dials-mystery       31          1695
                              32          1026
                              33          3541
                              34           454
the-tragedy-at-marsdon-manor  1           4492
Length: 325, dtype: int64

In [ ]:
# testing
CORPUS.groupby(bags[bag]).size().groupby('book_id').count()

book_id
giants-bread                                   29
the-adventure-of-the-cheap-flat                 1
the-adventure-of-the-egyptian-tomb              1
the-adventure-of-the-italian-nobleman           1
the-adventure-of-the-western-star               1
the-big-four                                   18
the-case-of-the-missing-will                    1
the-disappearance-of-mr-davenheim               1
the-jewel-robbery-at-the-grand-metropolitan     1
the-kidnapped-prime-minister                    1
the-man-in-the-brown-suit                      37
the-million-dollar-bond-robbery                 1
the-murder-at-the-vicarage                     32
the-murder-of-roger-ackroyd                    27
the-murder-on-the-links                        28
the-mysterious-affair-at-styles                13
the-mystery-of-hunters-lodge                    1
the-mystery-of-the-blue-train                  36
the-secret-adversary                           29
the-secret-of-chimneys                    

## Build BOW

Create a BOW with chapter as the bag.

In [18]:
# insert corpus_to_bow function from M05 Homework
def corpus_to_bow(df = CORPUS, bag = 'CHAPS'):
    '''
    Function that generates a bag of words BOW representation of the CORPUS table.
    Arguments: 
    - Tokens dataframe. Default is entire CORPUS table but can be a subset of it.
    - Choice of bag, i.e. `OHCO` level, such as book, chapter, or paragraph. Default is "CHAPS"
    Output: 
    - BOW table: A multiindex (bag levels + term_str) dataframe and a column count 'n'
    '''
    # grab column names for bag
    bag_cols = bags[bag]
    
    BOW = df.groupby(bag_cols + ['term_str']).term_str.count().to_frame('n')

    return BOW

In [ ]:
# create BOW with chapter as bag
BOW_chaps = corpus_to_bow(CORPUS, bag)

# BOW_chaps

n
book_id                      chap_num term_str      
giants-bread                 0        a           53
                                      about        3
                                      above        1
                                      absolute     1
                                      absolutely   1
...                                               ..
the-tragedy-at-marsdon-manor 1        yielded      1
                                      you         79
                                      young       11
                                      your        10
                                      yourself     1

[252614 rows x 1 columns]

## Build DTM

Unstack the BOW into DTM and replace nulls with 0s.

In [ ]:
# unstack BOW into DTM and replace nulls with 0s
DTM = BOW_chaps.unstack(fill_value=0)

# drop the 'n' top level column artifact that came over from BOW count
DTM.columns = DTM.columns.droplevel(0)

# DTM

term_str                               1  10  100  100000  1015  1019  102  \
book_id                      chap_num                                        
giants-bread                 0         0   0    0       0     0     0    0   
                             1         1   0    0       0     0     0    0   
                             2         0   0    0       0     0     0    0   
                             3         0   0    0       0     0     0    0   
                             4         0   0    0       0     0     0    0   
...                                   ..  ..  ...     ...   ...   ...  ...   
the-seven-dials-mystery      31        0   0    0       0     0     0    0   
                             32        3   0    0       0     0     0    0   
                             33        0   0    0       0     0     0    0   
                             34        0   0    0       0     0     0    0   
the-tragedy-at-marsdon-manor 1         0   0    0       0     0     0    0   

term_str                               1023  103  1030  ...  épatant  \
book_id                      chap_num                   ...            
giants-bread                 0            0    0     0  ...        0   
                             1            0    0     0  ...        0   
                             2            0    0     0  ...        0   
                             3            0    0     0  ...        0   
                             4            0    0     0  ...        0   
...                                     ...  ...   ...  ...      ...   
the-seven-dials-mystery      31           0    0     0  ...        0   
                             32           0    0     0  ...        0   
                             33           0    0     0  ...        0   
                             34           0    0     0  ...        0   
the-tragedy-at-marsdon-manor 1            0    0     0  ...        0   

term_str                               épatants  épouvantable  état  étre  \
book_id                      chap_num                                       
giants-bread                 0                0             0     0     0   
                             1                0             0     0     0   
                             2                0             0     0     0   
                             3                0             0     0     0   
                             4                0             0     0     0   
...                                         ...           ...   ...   ...   
the-seven-dials-mystery      31               0             0     0     0   
                             32               0             0     0     0   
                             33               0             0     0     0   
                             34               0             0     0     0   
the-tragedy-at-marsdon-manor 1                0             0     0     0   

term_str                               évidemment  évident  êtes  œuvre  ⁷₁₁  
book_id                      chap_num                                         
giants-bread                 0                  0        0     0      0    0  
                             1                  0        0     0      0    0  
                             2                  0        0     0      0    0  
                             3                  0        0     0      0    0  
                             4                  0        0     0      0    0  
...                                           ...      ...   ...    ...  ...  
the-seven-dials-mystery      31                 0        0     0      0    0  
                             32                 0        0     0      0    0  
                             33                 0        0     0      0    0  
                             34                 0        0     0      0    0  
the-tragedy-at-marsdon-manor 1                  0        0     0      0    0  

[325 rows x 20953 

## Build TFIDF

Use SKLearn to convert DTM into a TFIDF data frame by passing DTM to TfidfTransformer with defaults and then saving the results to a dataframe TFIDF that preserves the index and columns of DTM.

In [ ]:
# instantiate tfidf engine with defaults
tfidf_engine = TfidfTransformer()

# fit and transform the DTM and then convert output back into a dataframe
TFIDF = pd.DataFrame(
    tfidf_engine.fit_transform(DTM).toarray(), #TfidfTransformer() ouputs a sparse matrix so we use .toarray() to convert it to a dense one
    index = DTM.index,
    columns = DTM.columns
)

# TFIDF

term_str                                      1   10  100  100000  1015  1019  \
book_id                      chap_num                                           
giants-bread                 0         0.000000  0.0  0.0     0.0   0.0   0.0   
                             1         0.011515  0.0  0.0     0.0   0.0   0.0   
                             2         0.000000  0.0  0.0     0.0   0.0   0.0   
                             3         0.000000  0.0  0.0     0.0   0.0   0.0   
                             4         0.000000  0.0  0.0     0.0   0.0   0.0   
...                                         ...  ...  ...     ...   ...   ...   
the-seven-dials-mystery      31        0.000000  0.0  0.0     0.0   0.0   0.0   
                             32        0.085231  0.0  0.0     0.0   0.0   0.0   
                             33        0.000000  0.0  0.0     0.0   0.0   0.0   
                             34        0.000000  0.0  0.0     0.0   0.0   0.0   
the-tragedy-at-marsdon-manor 1         0.000000  0.0  0.0     0.0   0.0   0.0   

term_str                               102  1023  103  1030  ...  épatant  \
book_id                      chap_num                        ...            
giants-bread                 0         0.0   0.0  0.0   0.0  ...      0.0   
                             1         0.0   0.0  0.0   0.0  ...      0.0   
                             2         0.0   0.0  0.0   0.0  ...      0.0   
                             3         0.0   0.0  0.0   0.0  ...      0.0   
                             4         0.0   0.0  0.0   0.0  ...      0.0   
...                                    ...   ...  ...   ...  ...      ...   
the-seven-dials-mystery      31        0.0   0.0  0.0   0.0  ...      0.0   
                             32        0.0   0.0  0.0   0.0  ...      0.0   
                             33        0.0   0.0  0.0   0.0  ...      0.0   
                             34        0.0   0.0  0.0   0.0  ...      0.0   
the-tragedy-at-marsdon-manor 1         0.0   0.0  0.0   0.0  ...      0.0   

term_str                               épatants  épouvantable  état  étre  \
book_id                      chap_num                                       
giants-bread                 0              0.0           0.0   0.0   0.0   
                             1              0.0           0.0   0.0   0.0   
                             2              0.0           0.0   0.0   0.0   
                             3              0.0           0.0   0.0   0.0   
                             4              0.0           0.0   0.0   0.0   
...                                         ...           ...   ...   ...   
the-seven-dials-mystery      31             0.0           0.0   0.0   0.0   
                             32             0.0           0.0   0.0   0.0   
                             33             0.0           0.0   0.0   0.0   
                             34             0.0           0.0   0.0   0.0   
the-tragedy-at-marsdon-manor 1              0.0           0.0   0.0   0.0   

term_str                               évidemment  évident  êtes  œuvre  ⁷₁₁  
book_id                      chap_num                                         
giants-bread                 0                0.0      0.0   0.0    0.0  0.0  
                             1                0.0      0.0   0.0    0.0  0.0  
                             2                0.0      0.0   0.0    0.0  0.0  
                             3                0.0      0.0   0.0    0.0  0.0  
                             4                0.0      0.0   0.0    0.0  0.0  
...                                           ...      ...   ...    ...  ...  
the-seven-dials-mystery      31               0.0      0.0   0.0    0.0  0.0  
                             32               0.0      0.0   0.0    0.0  0.0  
                             33               0.0      0.0   0.0    0.0  0.0  
                             34               0.0      0.0   0.0    0.0  

## Reduce Feature Space (and compute DFIDF)

Reduce the feature space of DTM, i.e. the vocabulary, by filtering for words whose document frequency is greater than or equal to 50 and information is greater than or equal to 10.

I have chosen these same thresholds as the M07 HW to start with because there were 320 documents in that corpus and I have 325 in mine.

In [24]:
# calculate DF (document frequency)
df_count = (DTM > 0).sum()

# calculate I (information)
tf = DTM.sum()
p = tf / tf.sum()
information = -np.log2(p)

In [ ]:
# create term stats dataframe
TERM_STATS = pd.DataFrame({'df': df_count, 'i': information})

# TERM_STATS

,df,i
term_str,,
1,24,14.656108
10,2,18.700502
100,2,18.700502
100000,1,19.700502
1015,1,19.700502
...,...,...
évidemment,3,18.115540
évident,1,19.700502
êtes,1,19.700502


In [ ]:
# get index to filter for df >= 50 and i >= 10
keep_terms = TERM_STATS[(TERM_STATS['df'] >= 50) & (TERM_STATS['i'] >= 10)].index

# keep_terms

Index(['able', 'above', 'abruptly', 'absolutely', 'absurd', 'accepted',
       'account', 'across', 'act', 'actually',
       ...
       'wrong', 'wrote', 'yard', 'year', 'years', 'yesterday', 'yet', 'young',
       'yours', 'yourself'],
      dtype='str', name='term_str', length=961)

In [ ]:
# apply filter to TFIDF table
TFIDF_reduced = TFIDF[keep_terms]

# TFIDF_reduced

term_str                                   able     above  abruptly  \
book_id                      chap_num                                 
giants-bread                 0         0.000000  0.011197  0.000000   
                             1         0.000000  0.008970  0.000000   
                             2         0.000000  0.000000  0.000000   
                             3         0.000000  0.006453  0.000000   
                             4         0.006418  0.000000  0.005002   
...                                         ...       ...       ...   
the-seven-dials-mystery      31        0.007305  0.011388  0.000000   
                             32        0.000000  0.000000  0.022132   
                             33        0.012510  0.000000  0.000000   
                             34        0.000000  0.000000  0.000000   
the-tragedy-at-marsdon-manor 1         0.003535  0.000000  0.000000   

term_str                               absolutely    absurd  accepted  \
book_id                      chap_num                                   
giants-bread                 0           0.008829  0.010225  0.000000   
                             1           0.000000  0.000000  0.009151   
                             2           0.000000  0.000000  0.015260   
                             3           0.000000  0.000000  0.000000   
                             4           0.003945  0.009137  0.005103   
...                                           ...       ...       ...   
the-seven-dials-mystery      31          0.008980  0.000000  0.000000   
                             32          0.000000  0.000000  0.000000   
                             33          0.005126  0.000000  0.000000   
                             34          0.000000  0.000000  0.000000   
the-tragedy-at-marsdon-manor 1           0.008691  0.000000  0.005622   

term_str                               account    across       act  actually  \
book_id                      chap_num                                          
giants-bread                 0             0.0  0.006723  0.000000  0.000000   
                             1             0.0  0.005386  0.000000  0.000000   
                             2             0.0  0.017962  0.000000  0.000000   
                             3             0.0  0.000000  0.000000  0.000000   
                             4             0.0  0.000000  0.000000  0.004136   
...                                        ...       ...       ...       ...   
the-seven-dials-mystery      31            0.0  0.000000  0.000000  0.000000   
                             32            0.0  0.013289  0.000000  0.000000   
                             33            0.0  0.000000  0.000000  0.021501   
                             34            0.0  0.000000  0.000000  0.031898   
the-tragedy-at-marsdon-manor 1             0.0  0.003309  0.011169  0.000000   

term_str                               ...     wrong     wrote  yard  \
book_id                      chap_num  ...                             
giants-bread                 0         ...  0.007209  0.000000   0.0   
                             1         ...  0.000000  0.000000   0.0   
                             2         ...  0.019262  0.000000   0.0   
                             3         ...  0.000000  0.000000   0.0   
                             4         ...  0.000000  0.000000   0.0   
...                                    ...       ...       ...   ...   
the-seven-dials-mystery      31        ...  0.000000  0.000000   0.0   
                             32        ...  0.000000  0.000000   0.0   
                             33        ...  0.004186  0.005839   0.0   
                             34        ...  0.000000  0.000000   0.0   
the-tragedy-at-marsdon-manor 1         ...  0.000000  0.000000   0.0   

term_str                                   year     years  yesterday  \
book_id                      chap_num                                  
gian

## Normalize (TFIDF_L2)

## Update VOCAB with DFIDF

## Save Outputs